# 01 · Array Basics

Warm-up problems on 1-D arrays (Python lists): finding the largest values, checking order, removing duplicates, rotating, and moving zeros.

**Topics covered**
1. Largest / second largest element
2. Check if an array is sorted
3. Remove duplicates from a sorted array
4. Rotate an array (by 1 and by k places)
5. Move zeros to the end

> 💡 Every problem is solved with a **brute-force** idea first, then improved. Look at the `Time` / `Space` notes to see *why* the next version is better.

## 1. Largest element

**Idea:** keep a variable `largest` and update it while scanning the array once.

- Starting with `nums[0]` works only if the array is non-empty.
- Starting with `float("-inf")` is safer, because *any* number is bigger than `-inf` (it also works when all numbers are negative).

**Time:** O(n) · **Space:** O(1)

In [1]:
nums = [1, 5, 6, 88, 93, -44, 56, 23, 90]

# Version 1: start from the first element
largest = nums[0]
for i in range(len(nums)):
    largest = max(largest, nums[i])      # keep the bigger of the two
print("Largest (start = nums[0]):", largest)

# Version 2: start from -infinity (safe for any input)
largest = float("-inf")
for x in nums:
    largest = max(largest, x)
print("Largest (start = -inf):  ", largest)

Largest (start = nums[0]): 93
Largest (start = -inf):   93


## 2. Second largest element

**Approach A – two passes:** first find the largest, then find the biggest number that is *not equal* to it.

**Approach B – one pass:** keep both `largest` and `second` and update them together.
When a new number beats `largest`, the old `largest` becomes `second`.

**Time:** O(n) for both · **Space:** O(1)

In [2]:
nums = [1, 5, 6, 88, 93, -44, 56, 23, 90]

# Approach A: two passes
largest = float("-inf")
second = float("-inf")
for x in nums:
    largest = max(largest, x)
for x in nums:
    if x > second and x != largest:      # ignore the largest value itself
        second = x
print("Second largest (two passes):", second)

# Approach B: one pass
largest = second = float("-inf")
for x in nums:
    if x > largest:
        second = largest                 # old largest is demoted
        largest = x
    elif x > second and x != largest:    # between the two → new second
        second = x
print("Second largest (one pass): ", second)

Second largest (two passes): 90
Second largest (one pass):  90


## 3. Check if an array is sorted

An array is sorted (ascending) if **no element is bigger than the one after it**.
So we scan pairs `(nums[i], nums[i+1])` and return `False` the moment we see a pair out of order.

> 🐞 **Fix:** the earlier version used `nums[i] < nums[i+1]` which actually checks for *descending* order. For ascending order, the "bad" pair is `nums[i] > nums[i+1]`.

**Time:** O(n) · **Space:** O(1)

In [3]:
def is_sorted(nums):
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:        # found a pair out of order
            return False
    return True

print(is_sorted([1, 5, 6, 88, 93, -44, 56, 23, 90]))  # False
print(is_sorted([1, 2, 2, 5, 9]))                     # True (equal neighbours are fine)

False
True


## 4. Remove duplicates from a sorted array

Because the array is **sorted**, duplicates always sit next to each other.

| Approach | Idea | Time | Space |
|---|---|---|---|
| Brute force | collect unique values in a new list (`not in` check) | O(n²) | O(n) |
| Hash / dict | a dict keeps unique keys in insertion order | O(n) | O(n) |
| **Two pointers** | `i` marks the last unique value, `j` scans ahead | O(n) | **O(1)** |

In [4]:
# Brute force: build a new list with only unseen values
nums = [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10]
unique = []
for x in nums:
    if x not in unique:                  # linear search each time → O(n²)
        unique.append(x)
print("Brute force:", unique)

Brute force: [1, 2, 3, 4, 7, 9, 10]


In [5]:
# Hash approach: dict keys are unique and keep insertion order
freq = {}
for x in nums:
    freq[x] = 0
unique = list(freq.keys())
print("Dict       :", unique, "→ count =", len(unique))

Dict       : [1, 2, 3, 4, 7, 9, 10] → count = 7


In [6]:
# Two pointers (in-place): the optimal solution
nums = [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10]
i = 0                                    # i = index of the last unique element found
for j in range(1, len(nums)):            # j scans the rest of the array
    if nums[j] != nums[i]:               # a new value appears
        i += 1
        nums[i] = nums[j]                # place it right after the last unique one
print("Two pointers: first", i + 1, "elements →", nums[:i + 1])

Two pointers: first 7 elements → [1, 2, 3, 4, 7, 9, 10]


## 5. Rotate an array

### 5a. Right-rotate by 1 place
The last element jumps to the front and everyone else shifts one step right.

In [7]:
nums = [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10]
n = len(nums)

# Method 1: slicing  (creates a new list, so we assign back with nums[:] = ...)
a = nums.copy()
a[:] = [a[-1]] + a[:n - 1]
print("Slicing :", a)

# Method 2: shift manually from the back
b = nums.copy()
temp = b[n - 1]                          # save the last element
for i in range(n - 2, -1, -1):           # move elements right, starting from the end
    b[i + 1] = b[i]
b[0] = temp
print("Shifting:", b)

Slicing : [10, 1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9]
Shifting: [10, 1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9]


### 5b. Right-rotate by `k` places

| Approach | Time | Space |
|---|---|---|
| Repeat the "rotate by 1" k times | O(n·k) | O(1) |
| `pop()` + `insert(0, …)` k times | O(n·k) | O(1) |
| Slicing `nums[n-k:] + nums[:n-k]` | O(n) | O(n) |
| **Reversal algorithm** | O(n) | **O(1)** |

⚠️ Always do `k = k % n` first. Rotating by `n` gives back the same array, so `k = 12` on a 10-element array is the same as `k = 2`.

In [8]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
k = 5

# Brute force: rotate by 1, k times
nums = original.copy()
n = len(nums)
for _ in range(k % n):
    last = nums[n - 1]
    for i in range(n - 2, -1, -1):
        nums[i + 1] = nums[i]
    nums[0] = last
print("Brute force:", nums)

# pop / insert
nums = original.copy()
for _ in range(k % n):
    nums.insert(0, nums.pop())           # remove last, put it in front
print("pop/insert :", nums)

# Slicing
nums = original.copy()
r = k % n
nums[:] = nums[n - r:] + nums[:n - r]
print("Slicing    :", nums)

Brute force: [6, 7, 8, 9, 10, 1, 2, 3, 4, 5]
pop/insert : [6, 7, 8, 9, 10, 1, 2, 3, 4, 5]
Slicing    : [6, 7, 8, 9, 10, 1, 2, 3, 4, 5]


### 5c. Reversal algorithm (optimal, in place)

To rotate right by `k`:
1. Reverse the **last k** elements
2. Reverse the **first n−k** elements
3. Reverse the **whole** array

Example `[1,2,3,4,5,6,7]`, `k = 3`:
```
reverse last 3   → [1,2,3,4, 7,6,5]
reverse first 4  → [4,3,2,1, 7,6,5]
reverse all      → [5,6,7,1,2,3,4]  ✅
```

In [9]:
def reverse(nums, left, right):
    """Reverse nums[left..right] in place using two pointers."""
    while left < right:
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1

def rotate_right(nums, k):
    n = len(nums)
    if n == 0:
        return
    k %= n                               # handle k > n
    reverse(nums, n - k, n - 1)          # step 1: last k elements
    reverse(nums, 0, n - k - 1)          # step 2: first n-k elements
    reverse(nums, 0, n - 1)              # step 3: whole array

nums = [1, 2, 3, 4, 5, 6, 7]
rotate_right(nums, 3)
print(nums)                              # [5, 6, 7, 1, 2, 3, 4]

[5, 6, 7, 1, 2, 3, 4]


## 6. Move zeros to the end

Keep the **relative order** of the non-zero elements.

| Approach | Idea | Time | Space |
|---|---|---|---|
| Two new lists | one list for non-zeros, one for zeros, then join | O(n) | O(n) |
| Temp list | copy non-zeros to the front, fill the rest with 0 | O(n) | O(n) |
| **Two pointers** | `i` = first zero; swap each later non-zero into position `i` | O(n) | **O(1)** |

In [10]:
# Approach 1: two separate lists
nums = [1, 1, 0, 1, 2, 3, 0, 4, 4, 0, 7, 9, 9, 9, 10]
non_zero, zeros = [], []
for x in nums:
    (non_zero if x != 0 else zeros).append(x)
print("Two lists:", non_zero + zeros)

# Approach 2: temp list, then overwrite the original
nums = [1, 1, 0, 1, 2, 3, 0, 4, 4, 0, 7, 9, 9, 9, 10]
n = len(nums)
temp = [x for x in nums if x != 0]
for i in range(len(temp)):
    nums[i] = temp[i]
for i in range(len(temp), n):
    nums[i] = 0
print("Temp list:", nums)

Two lists: [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10, 0, 0, 0]
Temp list: [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10, 0, 0, 0]


In [11]:
# Approach 3: two pointers, in place
def move_zeros(nums):
    n = len(nums)
    # 1) find the first zero
    i = 0
    while i < n and nums[i] != 0:
        i += 1
    if i == n:                           # no zero at all → nothing to do
        return nums
    # 2) every non-zero after it is swapped forward
    for j in range(i + 1, n):
        if nums[j] != 0:
            nums[i], nums[j] = nums[j], nums[i]
            i += 1                       # i always points to the next zero slot
    return nums

# 🐞 Fix: the earlier version re-assigned `nums` inside the function, ignoring the argument.
print("Two pointers:", move_zeros([1, 1, 0, 1, 2, 3, 0, 4, 4, 0, 7, 9, 9, 9, 10]))
print("No zeros    :", move_zeros([1, 2, 3]))

Two pointers: [1, 1, 1, 2, 3, 4, 4, 7, 9, 9, 9, 10, 0, 0, 0]
No zeros    : [1, 2, 3]


---
### ✅ Key takeaways
- Use `float("-inf")` / `float("inf")` as safe starting values.
- **Two pointers** turn many O(n)-space solutions into O(1)-space ones.
- For rotation, reduce `k` with `k % n`, then use the **reversal trick**.